In [1]:
import torch
import os
import subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import snapshot_download, login
from datasets import load_dataset



In [2]:
from peft import PeftModel, PeftConfig
import json
import re
from IPython.display import Markdown, display
import numpy as np
import json
from tqdm import tqdm
from torch.nn.attention import SDPBackend, sdpa_kernel

In [3]:
# torch.backends.cuda.flash_sdp_enabled()

In [4]:
BASE_MODEL_DIR = "/data/Iman/shared_LLM_models/Qwen2.5-Math-7B/"
ADAPTER_DIR = "./finetuned_models/lora-qwen-atc-finetuned-lora-adapters/"
GRPO_CHECKPOINT = "/data/Iman/finetuned_models/grpo-qwen-atc/checkpoint-2000/"

## 1. Load tokenizer and model

In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# attach LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    GRPO_CHECKPOINT)

model = model.merge_and_unload()
model.eval()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

In [6]:
# model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params/1e6:.2f}M")
print(f"Trainable parameters: {trainable_params/1e6:.2f}M")

Total parameters: 7615.62M
Trainable parameters: 0.00M


In [7]:
# saving the merged model
merged_model_dir = "/data/Iman/finetuned_models/grpo-qwen-atc/merged_grpo_qwen_model/checkpoint-2000/"

model.save_pretrained(merged_model_dir)
tokenizer.save_pretrained(merged_model_dir)

('/data/Iman/finetuned_models/grpo-qwen-atc/merged_grpo_qwen_model/checkpoint-2000/tokenizer_config.json',
 '/data/Iman/finetuned_models/grpo-qwen-atc/merged_grpo_qwen_model/checkpoint-2000/special_tokens_map.json',
 '/data/Iman/finetuned_models/grpo-qwen-atc/merged_grpo_qwen_model/checkpoint-2000/vocab.json',
 '/data/Iman/finetuned_models/grpo-qwen-atc/merged_grpo_qwen_model/checkpoint-2000/merges.txt',
 '/data/Iman/finetuned_models/grpo-qwen-atc/merged_grpo_qwen_model/checkpoint-2000/added_tokens.json',
 '/data/Iman/finetuned_models/grpo-qwen-atc/merged_grpo_qwen_model/checkpoint-2000/tokenizer.json')

In [ ]:
# # load the merged model
# quantization_config = BitsAndBytesConfig(
#     load_in_16bit=True,
# )
# # model = AutoModelForCausalLM.from_pretrained(
# #     merged_model_dir,
# #     torch_dtype=torch.float16,
# #     device_map="auto",
# #     # quantization_config=quantization_config,
# #     # attn_implementation="sdpa",
# #     # trust_remote_code=True,
# # )
# model = AutoModelForCausalLM.from_pretrained(
#     merged_model_dir, # BASE_MODEL_DIR,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     # quantization_config=quantization_config,
#     # attn_implementation="sdpa",
#     # trust_remote_code=True,
# )

# model.eval()


## 2. Extracting Data and formating

In [8]:
# read the data from json file
file_path = 'ownship_prompts_data_Nov10.jsonl'
with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]

# ds = load_dataset('json', data_files=file_path, split='train').train_test_split(test_size=0.2, seed=42)
# test = ds['test']

def format_prompt(example):
    return (f"### System:\n{example['system_prompt'].strip()}\n\n"
            f"### User:\n{example['user_prompt'].strip()}\n\n"
            f"### Assistant:\n{example['prompt_response'].strip()}"
    )

In [9]:
len(data)

136267

In [10]:
data[0]  # Display the first two entries to verify the data structure

{'system_prompt': 'You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, \n    such as merging or intersection points, where other agents (intruders) may also be approaching. Based on the information of the intruders, \n    the ownship should take an action to avoid collisions. The ownship agent only has access to information about the front intruders, \n    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by \n    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.\n    Now, given the ownship state: [v, v_min, v_max, delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],\n    where:\n      v: ownship speed (m/s),\n      v_min: ownship minimum speed (m/s),\n      v_max: ownship maximum s

In [14]:
# data analysis: 1. count data with user prompt containing nan after 3 numbers in the "= [ "
nan_count = 0
actions_with_nan = [] # retrieve  from the prompt response 'Acelerate|Decelerate|Hold'
for example in data:
    user_prompt = example['user_prompt']
    pattern = r"=\s*\[\s*([-+]?\d*\.?\d+\s*,\s*[-+]?\d*\.?\d+\s*,\s*[-+]?\d*\.?\d+\s*,\s*nan,\s*nan)"
    if re.search(pattern, user_prompt):
        nan_count += 1
        # extract action
        response = example['prompt_response']
        action_match = re.search(r'\b(Accelerate|Decelerate|Hold)\b', response, re.IGNORECASE)
        if action_match:
            actions_with_nan.append(action_match.group(1).capitalize())
print(f"Number of examples with 'nan' after 3 numbers in the '= [ ]': {nan_count}")
print(f"Actions associated with 'nan' examples: {actions_with_nan}")

# count the number of occurrences of each action in actions_with_nan
from collections import Counter
action_counts = Counter(actions_with_nan)
print(f"Action counts for 'nan' examples: {action_counts}")

Number of examples with 'nan' after 3 numbers in the '= [ ]': 9858
Actions associated with 'nan' examples: ['Accelerate', 'Accelerate', 'Accelerate', 'Accelerate', 'Accelerate', 'Accelerate', 'Hold', 'Accelerate', 'Hold', 'Hold', 'Accelerate', 'Accelerate', 'Accelerate', 'Hold', 'Hold', 'Accelerate', 'Accelerate', 'Hold', 'Hold', 'Accelerate', 'Accelerate', 'Hold', 'Hold', 'Accelerate', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 'Accelerate', 'Hold', 'Accelerate', 'Hold', 'Accelerate', 'Hold', 'Accelerate', 'Hold', 'Accelerate', 'Hold', 'Accelerate', 'Accelerate', 'Hold', 'Accelerate', 'Accelerate', 'Hold', 'Hold', 'Accelerate', 'Hold', 'Hold', 'Accelerate', 'Hold', 'Hold', 'Hold', 'Hold', 'Hold', 

In [11]:
# actions distribution in the whole dataset
from collections import Counter
all_actions = []
for example in data:
    response = example['prompt_response']
    action_match = re.search(r'\b(Accelerate|Decelerate|Hold)\b', response, re.IGNORECASE)
    if action_match:
        all_actions.append(action_match.group(1).capitalize())
action_distribution = Counter(all_actions)
print(f"Overall action distribution: {action_distribution}")

Overall action distribution: Counter({'Hold': 83915, 'Accelerate': 37123, 'Decelerate': 15229})


In [12]:
ex_prompt = format_prompt(data[1000])
print(ex_prompt)  # Format the first entry to verify the prompt structure

### System:
You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, 
    such as merging or intersection points, where other agents (intruders) may also be approaching. Based on the information of the intruders, 
    the ownship should take an action to avoid collisions. The ownship agent only has access to information about the front intruders, 
    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    Now, given the ownship state: [v, v_min, v_max, delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
    where:
      v: ownship speed (m/s),
      v_min: ownship minimum speed (m/s),
      v_max: ownship maximum speed (m/s),
    

In [13]:
inputs = tokenizer(ex_prompt, return_tensors="pt").to("cuda")
inputs

{'input_ids': tensor([[14374,   739,   510,  2610,   525,   458, 87182, 38647,   409,  6135,
         21242,   290, 17847,    13,  2411,  1817,   882,  3019,    11,   458,
          1828,  5270,  8315,   374, 29947,   264, 87838, 65176,    11,   715,
           262,  1741,   438, 53377,   476, 19246,  3501,    11,  1380,  1008,
         13009,   320, 58788,   661,   388,     8,  1231,  1083,   387, 29947,
            13, 20205,   389,   279,  1995,   315,   279, 10582,   661,   388,
            11,   715,   262,   279,  1828,  5270,  1265,  1896,   458,  1917,
           311,  5648, 47353,    13,   576,  1828,  5270,  8315,  1172,   702,
          2615,   311,  1995,   911,   279,  4065, 10582,   661,   388,    11,
           715,   262,   714,  1052,  2578,   387,  1008, 10582,   661,   388,
          4815,   279,  1828,  5270,    13,  4615,  3383,   374,   311,  1492,
           279,  1828,  5270, 14135,  5648, 47353,   448,  4065, 10582, 32619,
         14135,   553,   715,   262, 2

In [14]:
with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    output_tokens = base_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        temperature=0.0,
        top_p=0.95,
        # repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

response_text = tokenizer.batch_decode(output_tokens[0], skip_special_tokens=True)
# print("Full Response:")
response_text = "".join(response_text)
# print(response_text)
assistant_output = response_text.split("### Assistant:")[-1].strip()
print("Assistant's Response:")
print(assistant_output)

/home/alexzongo/miniconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/alexzongo/miniconda3/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Assistant's Response:
The recommended action for the ownship is to 'Hold'.


In [15]:
# locate the word "Accelerate|Hold|Decelerate" in the response
match = re.search(r'\b(Accelerate|Hold|Decelerate)\b', assistant_output, re.IGNORECASE)
if match:
    predicted_action = match.group(1).capitalize()
    print(f"Predicted Action: {predicted_action}")

Predicted Action: Hold


## 3. Accuracy or batch evaluation

In [16]:
# test data = randomly sample 1000 entries from data
import random
test_data = random.sample(data, 1000)
test_data

[{'system_prompt': 'You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, \n    such as merging or intersection points, where other agents (intruders) may also be approaching. Based on the information of the intruders, \n    the ownship should take an action to avoid collisions. The ownship agent only has access to information about the front intruders, \n    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by \n    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.\n    Now, given the ownship state: [v, v_min, v_max, delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],\n    where:\n      v: ownship speed (m/s),\n      v_min: ownship minimum speed (m/s),\n      v_max: ownship maximum 

In [17]:
def retrieve_action(response):
    match = re.search(r'\b(Accelerate|Hold|Decelerate)\b', response, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return None

In [ ]:
def generate_response(system_prompt, user_prompt):
    prompt = f"### System:\n{system_prompt.strip()}\n\n### User:\n{user_prompt.strip()}\n\n### Assistant:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        output_tokens = base_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    decoded = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    decoded = "".join(decoded)
    return decoded.split("### Assistant:")[-1].strip()



correct, total = 0, 0
for ex in tqdm(test_data):
    pred_response = generate_response(ex['system_prompt'], ex['user_prompt'])
    # the last word of the response should match "Acelerate|Hold|Decelerate"
    # so we can search for that word in the response and compare

    if pred_response.lower() == ex['prompt_response'].strip().lower():
        correct += 1
    total += 1

accuracy = correct / total
print(f"Accuracy over {total} samples: {accuracy:.4f}")

In [ ]:
def build_prompt(example):
    return (f"### System:\n{example['system_prompt'].strip()}\n\n"
            f"### User:\n{example['user_prompt'].strip()}\n\n"
            f"### Assistant:\n"
    )

def generate_batch_responses(examples):
    responses = []
    prompts = [build_prompt(example) for example in examples]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        output_tokens = base_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            # top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            # use_cache=True,
        )

    decoded = tokenizer.batch_decode(output_tokens[0], skip_special_tokens=True)
    # decoded = "".join(decoded)
    # assistant_response = decoded.split("### Assistant:")[-1].strip()
    # responses.append(assistant_response)
    return decoded

decoded = generate_batch_responses(test_data[:4])
decoded
# for i, response in enumerate(decoded):
#     assistant_response = response.split("### Assistant:")[-1].strip()
#     print(f"Response {i+1}: {assistant_response}")

In [ ]:
import math

def compute_perplexity(model, tokenizer, dataset):
    losses = []
    for ex in dataset:
        prompt = f"### System:\n{ex['system_prompt']}\n\n### User:\n{ex['user_prompt']}\n\n### Assistant:\n{ex['prompt_response']}"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
        with torch.no_grad():
            loss = model(**inputs, labels=inputs["input_ids"]).loss
        losses.append(loss.item())
    return math.exp(np.mean(losses))

perplexity = compute_perplexity(model, tokenizer, test_data)
print(f"Perplexity on test data: {perplexity:.4f}")

## Batched evaluation (not working)

In [ ]:
data_loaded = load_dataset('json', data_files=file_path, split='train').train_test_split(test_size=0.005, seed=42)
train_data = data_loaded['train']
test_data = data_loaded['test']

In [ ]:
test_data

In [ ]:
test_data[0:10]

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 4
MAX_LEN = 1024


def collate_fn(examples):
    prompts = [build_prompt(ex) for ex in examples]
    # print(prompts[0])
    refs = [ex['prompt_response'] for ex in examples]
    encodings = tokenizer(prompts, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding=True)
    encodings["prompts"] = prompts # keep raw text for decoding later
    encodings["refs"] = refs
    return encodings

dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
predictions, references = [], []

for batch in tqdm(dataloader, desc="Evaluating"):
    input_ids = batch['input_ids'].to("cuda")
    attention_mask = batch['attention_mask'].to("cuda")
    prompts = batch['prompts']
    refs = batch['refs']

    # print(batch)
    with torch.no_grad():
        output_tokens = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)
    for text, ex in zip(decoded, prompts):
        assistant_response = text.split("### Assistant:")[-1].strip()
        predictions.append(assistant_response)

    references.extend(refs)

In [ ]:
len(predictions), len(references)

In [ ]:
predictions[0:5], references[0:5]

In [ ]:
def accuracy_score(preds, refs):
    correct = sum(p.strip().lower() == r.strip().lower() for p, r in zip(preds, refs))
    return correct / len(refs)

acc = accuracy_score(predictions, references)
print(f"Batched Evaluation Accuracy: {acc:.4f}")